# 02 DPO
Set DATASET to hh or ultrafeedback and run once per dataset.
Attach: repo zip dataset, 01_sft notebook output. Accelerator: GPU T4 x2. Internet: on.

In [ ]:
import glob, os, shutil, zipfile
matches = glob.glob("/kaggle/input/**/submission_adapter.py", recursive=True)
if not matches:
    for archive in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        zipfile.ZipFile(archive).extractall("/tmp/unzipped")
    matches = glob.glob("/tmp/unzipped/**/submission_adapter.py", recursive=True)
assert matches, "attach the repo zip as a Kaggle dataset"
REPO = "/tmp/repo"
shutil.rmtree(REPO, ignore_errors=True)
shutil.copytree(os.path.dirname(matches[0]), REPO)
os.chdir(REPO)
os.environ["PYTHONPATH"] = REPO
OUT = "/kaggle/working/outputs"
os.makedirs(OUT, exist_ok=True)
print(os.listdir(REPO))

In [ ]:
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest
!python -m pip install -q -e . --no-deps

In [ ]:
!nvidia-smi
import torch, transformers, datasets
print("torch", torch.__version__, "transformers", transformers.__version__, "datasets", datasets.__version__)
print("gpus", torch.cuda.device_count())

In [ ]:
DATASET = "hh"

In [ ]:
def find(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}/config.json", recursive=True))
    assert hits, f"no checkpoint matching {pattern} under /kaggle/input"
    return os.path.dirname(hits[0])

SFT_BEST = find("sft/best")
print(SFT_BEST)

## Data

In [ ]:
DATA = f"{OUT}/data/dpo"
!python scripts/prepare_dpo_data.py --dataset {DATASET} --output_dir {DATA}

## Smoke test (2 steps)

In [ ]:
!python -m alignment.train_dpo --policy_path {SFT_BEST} --train_path {DATA}/{DATASET}_train.jsonl --val_path {DATA}/{DATASET}_val.jsonl --output_dir /tmp/smoke_dpo --max_train_examples 64 --max_val_examples 16 --max_steps 2 --eval_every 2 --log_every 1

## Full run

In [ ]:
!python -m alignment.train_dpo --policy_path {SFT_BEST} --train_path {DATA}/{DATASET}_train.jsonl --val_path {DATA}/{DATASET}_val.jsonl --output_dir {OUT}/dpo_{DATASET}

In [ ]:
!cd /kaggle/working && zip -rq dpo_{DATASET}_run_outputs.zip outputs/dpo_{DATASET} outputs/data
!ls -lh /kaggle/working